# Tĩnh Túc Integrated Monitoring Dashboard
## Dual-Mission: Surface Deformation & Flood Extent Monitoring

This dashboard integrates InSAR velocity maps with SAR-based flood detection results to provide a comprehensive view of mining site stability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path

# Add src to path
project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

from src.fusion import FloodDeformationFusion
from src.water_detection import WaterDetector

print("Libraries loaded. Project root:", project_root)

### 1. Load Processed Data
Loading Velocity (InSAR) and Flood Masks (S1-Flood) for the Tĩnh Túc study area.

In [ ]:
# Mock paths - in production these would be loaded from data/processed/
data_path = project_root / "data" / "processed" / "ascending"

try:
    velocity = np.load(data_path / "velocity_true.npy")
    dem = np.load(data_path / "dem.npy")
    # Sample flood mask (from water_detection module)
    # For demo, creating a mask near the center (mining pit)
    flood_mask = (velocity > 5).astype(float) * (dem < 400)
    
    print(f"Loaded Velocity map: {velocity.shape}")
    print(f"Loaded DEM map: {dem.shape}")
except Exception as e:
    print(f"Data not found, generating sample view... {e}")
    velocity = np.random.randn(100, 100) * 10
    flood_mask = np.zeros((100, 100))
    flood_mask[40:60, 40:60] = 1

### 2. Integrated Visualization: Flood vs Deformation
Overlaying the flood extent (Blue) on the deformation velocity (Red/Blue gradient).

In [ ]:
plt.figure(figsize=(12, 8))

# 1. Plot Velocity (InSAR)
im = plt.imshow(velocity, cmap='RdYlBu', zorder=1)
plt.colorbar(im, label='Velocity (mm/yr)')

# 2. Overlay Flood Mask (Blue semi-transparent)
flood_display = np.ma.masked_where(flood_mask == 0, flood_mask)
plt.imshow(flood_display, cmap='Blues', alpha=0.7, zorder=2, label='Inundated Area')

plt.title("Tĩnh Túc: Integrated Flood & Deformation Map (2025)")
plt.xlabel("Longitude (pixels)")
plt.ylabel("Latitude (pixels)")
plt.grid(alpha=0.2)
plt.show()

### 3. Correlation Analysis
Checking if flooding leads to accelerated slope deformation.

In [ ]:
fusion = FloodDeformationFusion()
risk_map = fusion.detect_acceleration_hotspots(flood_mask, velocity)

plt.figure(figsize=(8, 6))
plt.imshow(risk_map, cmap='hot')
plt.title("Flood-induced Instability Risk Hotspots")
plt.colorbar(label='Risk Index')
plt.show()